# T3 SigLIP — full Flickr30k decoder adaptation seed 42

Full run được phát hành sau pilot đạt gate. Ba notebook T1/T2/T3 độc lập và có
thể chạy song song. T1/T2 chỉ dùng Validation để báo cáo zero-shot transfer,
không chọn checkpoint; T3 dùng Validation CIDEr-D để chọn decoder. Held-out
Test luôn niêm phong.

## Kaggle input bắt buộc — đúng 8 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_T3_Handoff`.
4. Output `phase3_teacher_c_seed42`.
5. Dataset `Flickr Image dataset`.
6. Dataset/Kaggle Model SigLIP offline có `MODEL_SOURCE_MANIFEST.json`.
7. Output `F30K_T3_Adapt42_Pilot`.
8. Package/dataset `pycocoevalcap-1.2` offline.

Khi resume, gắn thêm đúng một output gần nhất của chính notebook này.

## Runtime

- Accelerator: GPU T4; một GPU là đủ.
- Internet: Off.
- Train khoảng 0,70 giờ/epoch chưa tính Validation; tối đa 20 epoch, Early Stopping từ epoch 5, patience 4. Notebook hỗ trợ resume giữa epoch.


## Cell 1 — Full-run authorization và Flickr30k contract

In [1]:

import os
EXPERIMENT_ID = "F30K_T3_Adapt42_Full"
TEACHER_ID = "T3"
HANDOFF_RESULT_NAME = "F30K_T3_Handoff_RESULT.json"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import csv, hashlib, json, math, os, random, sys, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def image_index(required_names):
    candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in required_names]
    by_name = {}
    duplicate_aliases = 0
    for path in candidates:
        old = by_name.get(path.name)
        if old is None:
            by_name[path.name] = path
            continue
        if old.resolve() == path.resolve():
            continue
        assert old.stat().st_size == path.stat().st_size, (
            "Ảnh trùng tên nhưng khác nội dung/kích thước", path.name, old, path
        )
        duplicate_aliases += 1
        by_name[path.name] = min(
            (old, path), key=lambda p: (len(p.parts), len(str(p)), str(p))
        )
    missing = sorted(required_names - set(by_name))
    assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])
    return by_name, duplicate_aliases

DATA_PATH = unique_named("f30k_train.parquet")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")
HANDOFF_PATH = unique_named(HANDOFF_RESULT_NAME)

data_contract = read_json(DATA_CONTRACT_PATH)
base_ready = read_json(BASE_READY_PATH)
vocab = read_json(VOCAB_PATH)
handoff = read_json(HANDOFF_PATH)

assert data_contract["status"] == "complete"
assert data_contract["train_image_count"] == 29000
assert data_contract["validation_image_count"] == 1014
assert data_contract["test_image_count"] == 1000
assert data_contract["vocabulary_size"] == 7456
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False
assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert handoff["teacher_id"] == TEACHER_ID
assert handoff["flickr30k"]["test_set_status"] == "sealed"
assert handoff["flickr30k"]["test_set_used"] is False
assert sha256(DATA_PATH) == data_contract["artifacts"]["train"]["sha256"]
assert sha256(VOCAB_PATH) == data_contract["artifacts"]["vocabulary"]["sha256"]

train_df = pd.read_parquet(DATA_PATH)
assert train_df["image_id"].nunique() == 29000
assert len(train_df) == 145000
train_ids = sorted(int(x) for x in train_df["image_id"].unique())
required_names = {f"{image_id}.jpg" for image_id in train_ids}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(required_names)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), "Notebook pilot yêu cầu GPU T4."
DEVICE = torch.device("cuda:0")

print("Common gate             : PASSED")
print("Experiment              :", EXPERIMENT_ID)
print("Teacher                 :", TEACHER_ID)
print("Flickr30k Train         : 29,000 images")
print("Vocabulary              : 7,456")
print("Nested image aliases    :", DUPLICATE_ALIASES, "(deduplicated)")
print("Held-out Test           : SEALED / NOT READ")


Common gate             : PASSED
Experiment              : F30K_T3_Adapt42_Full
Teacher                 : T3
Flickr30k Train         : 29,000 images
Vocabulary              : 7,456
Nested image aliases    : 29000 (deduplicated)
Held-out Test           : SEALED / NOT READ


## Cell 2 — Production workload

In [2]:

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from transformers import SiglipModel

assert handoff["status"] == "passed_with_required_adaptation"
assert handoff["method_contract"]["source_vocabulary_size"] == 10080
assert handoff["method_contract"]["target_vocabulary_size"] == 7456
EXPECTED_CHECKPOINT_SHA = handoff["teacher_source"]["development_checkpoint_sha256"]
CHECKPOINT_PATH = unique_named("teacher_c_seed42_best.pt")
assert sha256(CHECKPOINT_PATH) == EXPECTED_CHECKPOINT_SHA

model_sources = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(manifest_path)
    except Exception:
        continue
    if payload.get("repo_id") == "google/siglip-base-patch16-224":
        model_sources.append(manifest_path.parent)
assert len(model_sources) == 1, model_sources
MODEL_SOURCE = model_sources[0]

transform = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
groups = []
for image_id, group in train_df.groupby("image_id", sort=True):
    groups.append((int(image_id), [list(map(int, x)) for x in group.token_ids]))

class TrainPilot(Dataset):
    def __len__(self): return len(groups)
    def __getitem__(self, index):
        image_id, captions = groups[index]
        tokens = captions[(image_id + index) % len(captions)]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
            pixels = transform(image.convert("RGB"))
        return pixels, torch.tensor(tokens, dtype=torch.long)

def collate(rows):
    images, captions = zip(*rows)
    maximum = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), maximum, dtype=torch.long)
    mask = torch.zeros(len(captions), maximum, dtype=torch.bool)
    for index, values in enumerate(captions):
        tokens[index, :len(values)] = values
        mask[index, :len(values)] = True
    return torch.stack(images), tokens, mask

class SiglipVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(torch.zeros(1, 196, 256))
        self.output_norm = nn.LayerNorm(256)
    def forward(self, patches):
        assert patches.shape[1:] == (196, 768)
        return self.output_norm(self.projection(patches) + self.position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(
            256, 8, 1024, 0.1, "gelu", batch_first=True, norm_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
        nn.init.normal_(self.token_embedding.weight, std=256 ** -0.5)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad(): self.token_embedding.weight[0].zero_()
    def forward(self, ids, memory, attention_mask=None):
        n, t = ids.shape
        positions = torch.arange(t, device=ids.device).unsqueeze(0).expand(n, -1)
        target = self.embedding_dropout(
            self.token_embedding(ids) * 16.0 + self.position_embedding(positions)
        )
        causal = torch.triu(
            torch.ones(t, t, dtype=torch.bool, device=ids.device), diagonal=1
        )
        padding = None if attention_mask is None else ~attention_mask.bool()
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal, tgt_key_padding_mask=padding
        )
        return F.linear(self.final_norm(hidden), self.token_embedding.weight, self.output_bias)

full_model = SiglipModel.from_pretrained(str(MODEL_SOURCE), local_files_only=True)
encoder = full_model.vision_model
del full_model
adapter = SiglipVisualAdapter()
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
assert checkpoint["schema_version"] == "phase3_teacher_c_seed42_checkpoint_v1"
encoder_state = encoder.state_dict()
encoder_state.update(checkpoint["encoder_candidate_state"])
encoder.load_state_dict(encoder_state, strict=True)
adapter.load_state_dict(checkpoint["adapter_state"], strict=True)
del checkpoint

torch.manual_seed(42)
decoder = CaptionDecoder(7456)
encoder, adapter, decoder = encoder.to(DEVICE), adapter.to(DEVICE), decoder.to(DEVICE)
encoder.eval(); adapter.eval(); decoder.train()
for module in (encoder, adapter):
    for parameter in module.parameters(): parameter.requires_grad_(False)



[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

## Cell 3 — Production workload

In [3]:

import importlib.util, shutil, subprocess
from torch.utils.data import DataLoader, Dataset

PILOT_RESULT_PATH = unique_named("F30K_T3_Adapt42_Pilot_RESULT.json")
pilot_result = read_json(PILOT_RESULT_PATH)
assert pilot_result["status"] == "passed"
assert pilot_result["pilot_gate"] == "full_decoder_adaptation_authorized"
assert pilot_result["pilot_weights_discarded"] is True
assert pilot_result["next_phase_authorized"] == EXPERIMENT_ID
assert pilot_result["logit_shape_verified"] is True

VAL_PATH = unique_named("f30k_validation.parquet")
PROTOCOL_PATH = unique_named("F30K_BASELINE_PROTOCOL.json")
assert sha256(VAL_PATH) == data_contract["artifacts"]["validation"]["sha256"]
protocol = read_json(PROTOCOL_PATH)
assert protocol["maximum_epochs"] == 20
assert protocol["selection_metric"] == "Validation CIDEr-D"

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))
assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR yêu cầu Java runtime."

val_df = pd.read_parquet(VAL_PATH)
assert val_df["image_id"].nunique() == 1014 and len(val_df) == 5070
all_names = required_names | {f"{int(x)}.jpg" for x in val_df["image_id"].unique()}
IMAGE_BY_NAME, DUPLICATE_ALIASES = image_index(all_names)

VAL_REFERENCES = {}
val_groups = []
for image_id, group in val_df.groupby("image_id", sort=True):
    image_id = int(image_id)
    val_groups.append((image_id, [list(map(int, x)) for x in group.token_ids]))
    VAL_REFERENCES[image_id] = [str(x) for x in group.caption.tolist()]

class TrainFull(Dataset):
    def __init__(self): self.epoch = 0
    def set_epoch(self, epoch): self.epoch = int(epoch)
    def __len__(self): return len(groups)
    def __getitem__(self, index):
        image_id, captions = groups[index]
        tokens = captions[(self.epoch + image_id % len(captions)) % len(captions)]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
            pixels = transform(image.convert("RGB"))
        return image_id, pixels, torch.tensor(tokens, dtype=torch.long)

class ValidationFull(Dataset):
    def __len__(self): return len(val_groups)
    def __getitem__(self, index):
        image_id, captions = val_groups[index]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
            pixels = transform(image.convert("RGB"))
        return image_id, pixels, torch.tensor(captions[0], dtype=torch.long)

def full_collate(rows):
    ids, images, captions = zip(*rows)
    maximum = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), maximum, dtype=torch.long)
    mask = torch.zeros(len(captions), maximum, dtype=torch.bool)
    for index, values in enumerate(captions):
        tokens[index, :len(values)] = values
        mask[index, :len(values)] = True
    return torch.tensor(ids), torch.stack(images), tokens, mask

train_set = TrainFull()
val_set = ValidationFull()
def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    return DataLoader(
        train_set, batch_size=8, shuffle=True,
        generator=torch.Generator().manual_seed(42 + epoch),
        num_workers=2, pin_memory=True, collate_fn=full_collate,
    )
val_loader = DataLoader(
    val_set, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True, collate_fn=full_collate,
)

TOKENS = vocab["tokens"]
SPECIAL = {0, 1, 2, 3}
def ids_to_caption(ids):
    words = []
    for value in ids:
        value = int(value)
        if value == 2: break
        if value not in SPECIAL: words.append(TOKENS[value])
    text = " ".join(words)
    for punctuation in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + punctuation, punctuation)
    return text.strip()

@torch.inference_mode()
def beam_decode(memory, beam_size=3, alpha=0.7):
    beams = [([1], 0.0, False)]
    for _ in range(30):
        active = [item for item in beams if not item[2]]
        if not active: break
        sequences = torch.tensor([x[0] for x in active], device=DEVICE)
        expanded = memory.expand(len(active), -1, -1)
        logits = decoder(sequences, expanded)[:, -1].float()
        logits[:, 0] = -torch.inf
        logits[:, 1] = -torch.inf
        log_probs = logits.log_softmax(-1)
        candidates = [item for item in beams if item[2]]
        for index, (tokens, score, _) in enumerate(active):
            values, indices = log_probs[index].topk(beam_size)
            for value, token in zip(values.tolist(), indices.tolist()):
                candidate = tokens + [token]
                candidates.append((candidate, score + value, token == 2))
        beams = sorted(
            candidates,
            key=lambda x: x[1] / (max(1, len(x[0]) - 1) ** alpha),
            reverse=True,
        )[:beam_size]
    return max(
        beams, key=lambda x: x[1] / (max(1, len(x[0]) - 1) ** alpha)
    )[0]

@torch.inference_mode()
def validate():
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    encoder.eval(); adapter.eval(); decoder.eval()
    hypotheses = {}
    ce_sum, token_count = 0.0, 0
    for image_ids, images, tokens, mask in val_loader:
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            patches = encoder(pixel_values=images).last_hidden_state
            memory = adapter(patches)
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
        targets, valid = tokens[:, 1:], mask[:, 1:]
        losses = F.cross_entropy(
            logits.float().reshape(-1, 7456), targets.reshape(-1), reduction="none"
        ).reshape_as(targets)
        ce_sum += float((losses * valid).sum())
        token_count += int(valid.sum())
        for index, image_id in enumerate(image_ids.tolist()):
            hypotheses[int(image_id)] = [ids_to_caption(beam_decode(memory[index:index+1]))]
    references = {key: [{"caption": x} for x in values] for key, values in VAL_REFERENCES.items()}
    predicted = {key: [{"caption": values[0]}] for key, values in hypotheses.items()}
    tokenizer = PTBTokenizer()
    references = tokenizer.tokenize(references)
    predicted = tokenizer.tokenize(predicted)
    bleu, _ = Bleu(4).compute_score(references, predicted)
    meteor = Meteor()
    try:
        meteor_value, _ = meteor.compute_score(references, predicted)
    finally:
        close = getattr(meteor, "close", None)
        if callable(close): close()
    rouge, _ = Rouge().compute_score(references, predicted)
    cider, _ = Cider().compute_score(references, predicted)
    return {
        "ce_loss": ce_sum / token_count, "BLEU_1": bleu[0],
        "BLEU_4": bleu[3], "METEOR": meteor_value,
        "ROUGE_L": rouge, "CIDEr_D": cider,
    }, hypotheses

MAX_EPOCHS = 20
PATIENCE = 4
MIN_EPOCHS = 5
MIN_DELTA = 0.001
ACCUMULATION = 8
MICRO_BATCH = 8
STEPS_PER_EPOCH = math.ceil(math.ceil(len(train_set) / MICRO_BATCH) / ACCUMULATION)
TOTAL_STEPS = MAX_EPOCHS * STEPS_PER_EPOCH
WARMUP = max(1, round(0.05 * TOTAL_STEPS))
optimizer = torch.optim.AdamW(
    decoder.parameters(), lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4
)
def lr_factor(step):
    if step < WARMUP: return max(1e-8, step / WARMUP)
    progress = min(1.0, (step - WARMUP) / max(1, TOTAL_STEPS - WARMUP))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)

LAST = WORK_ROOT / "last.pt"
BEST = WORK_ROOT / "best.pt"
METRICS = WORK_ROOT / "metrics.csv"
CAPTIONS = WORK_ROOT / "best_validation_captions.json"

def state_cpu(module):
    return {key: value.detach().cpu() for key, value in module.state_dict().items()}

def save_last(progress, history, best_cider, stale):
    payload = {
        "schema_version": "f30k_t3_adapt42_v1",
        "experiment_id": EXPERIMENT_ID, "seed": 42,
        "decoder_state_dict": state_cpu(decoder),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "progress": progress, "history": history,
        "best_cider": best_cider, "stale_evaluations": stale,
        "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
        "vocabulary_sha256": sha256(VOCAB_PATH),
        "test_set_status": "sealed", "test_set_used": False,
    }
    temporary = LAST.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, LAST)

def save_best(epoch, metrics):
    temporary = BEST.with_suffix(".tmp")
    torch.save({
        "schema_version": "f30k_t3_adapt42_best_v1",
        "experiment_id": EXPERIMENT_ID, "seed": 42, "epoch": epoch,
        "decoder_state_dict": state_cpu(decoder), "metrics": metrics,
        "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
        "vocabulary_sha256": sha256(VOCAB_PATH),
        "test_set_status": "sealed", "test_set_used": False,
    }, temporary)
    os.replace(temporary, BEST)

start_epoch, cursor, global_step = 0, 0, 0
loss_sum, token_count = 0.0, 0
history, best_cider, stale = [], -math.inf, 0
resume_candidates = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}/last.pt"))
if resume_candidates:
    assert len(resume_candidates) == 1, resume_candidates
    payload = torch.load(resume_candidates[0], map_location="cpu", weights_only=False)
    assert payload["experiment_id"] == EXPERIMENT_ID
    assert payload["teacher_checkpoint_sha256"] == sha256(CHECKPOINT_PATH)
    assert payload["vocabulary_sha256"] == sha256(VOCAB_PATH)
    assert payload["test_set_used"] is False
    decoder.load_state_dict(payload["decoder_state_dict"], strict=True)
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    scheduler.load_state_dict(payload["scheduler_state_dict"])
    scaler.load_state_dict(payload["scaler_state_dict"])
    progress = payload["progress"]
    start_epoch = int(progress["epoch"])
    cursor = int(progress["micro_cursor"])
    global_step = int(progress["global_step"])
    loss_sum = float(progress["loss_sum"])
    token_count = int(progress["token_count"])
    history = payload["history"]
    best_cider = float(payload["best_cider"])
    stale = int(payload["stale_evaluations"])
    resume_root = resume_candidates[0].parent
    shutil.copy2(resume_candidates[0], LAST)
    for name, destination in [
        ("best.pt", BEST),
        ("metrics.csv", METRICS),
        ("best_validation_captions.json", CAPTIONS),
    ]:
        source = resume_root / name
        if source.is_file():
            shutil.copy2(source, destination)
    print("Exact resume            :", resume_candidates[0])
    print("Epoch / micro cursor    :", start_epoch + 1, "/", cursor)

run_started = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    loader = make_train_loader(epoch)
    decoder.train(); encoder.eval(); adapter.eval()
    optimizer.zero_grad(set_to_none=True)
    current_cursor = cursor if epoch == start_epoch else 0
    if current_cursor == 0:
        loss_sum, token_count = 0.0, 0
    for micro_index, (_, images, tokens, mask) in enumerate(loader):
        if micro_index < current_cursor:
            continue
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            patches = encoder(pixel_values=images).last_hidden_state
            memory = adapter(patches)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
            targets, valid = tokens[:, 1:], mask[:, 1:]
            raw = F.cross_entropy(
                logits.float().reshape(-1, 7456), targets.reshape(-1), reduction="none"
            ).reshape_as(targets)
            loss = (raw * valid).sum() / valid.sum()
        assert logits.shape[-1] == 7456 and torch.isfinite(loss)
        scaler.scale(loss / ACCUMULATION).backward()
        loss_sum += float((raw.detach() * valid).sum())
        token_count += int(valid.sum())
        final_micro = micro_index + 1 == len(loader)
        if (micro_index + 1) % ACCUMULATION == 0 or final_micro:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
            step_in_epoch = math.ceil((micro_index + 1) / ACCUMULATION)
            if step_in_epoch % 100 == 0:
                save_last({
                    "epoch": epoch, "micro_cursor": micro_index + 1,
                    "global_step": global_step, "loss_sum": loss_sum,
                    "token_count": token_count,
                }, history, best_cider, stale)
                print(
                    f"epoch={epoch+1:02d} | step={step_in_epoch:04d}/{STEPS_PER_EPOCH} | "
                    f"CE={loss_sum/token_count:.6f}", flush=True
                )
    metrics, hypotheses = validate()
    metrics.update({
        "epoch": epoch + 1, "train_ce": loss_sum / token_count,
        "global_step": global_step,
        "elapsed_hours": (time.time() - run_started) / 3600,
    })
    history.append(metrics)
    pd.DataFrame(history).to_csv(METRICS, index=False)
    improved = metrics["CIDEr_D"] > best_cider + MIN_DELTA
    if improved:
        best_cider, stale = metrics["CIDEr_D"], 0
        save_best(epoch + 1, metrics)
        CAPTIONS.write_text(
            json.dumps({"references": VAL_REFERENCES, "hypotheses": hypotheses}, ensure_ascii=False),
            encoding="utf-8",
        )
    else:
        stale += 1
    save_last({
        "epoch": epoch + 1, "micro_cursor": 0, "global_step": global_step,
        "loss_sum": 0.0, "token_count": 0,
    }, history, best_cider, stale)
    print(
        f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | train CE={metrics['train_ce']:.6f} | "
        f"val CE={metrics['ce_loss']:.6f} | CIDEr-D={metrics['CIDEr_D']:.6f} | "
        f"stale={stale}/{PATIENCE}", flush=True
    )
    cursor = 0
    if epoch + 1 >= MIN_EPOCHS and stale >= PATIENCE:
        print("Early stopping          : TRIGGERED")
        break

assert BEST.is_file() and LAST.is_file() and METRICS.is_file() and CAPTIONS.is_file()
best_payload = torch.load(BEST, map_location="cpu", weights_only=False)
DIRECT_LOGIT_CONTRACT = {
    "schema_version": "f30k_t3s3_direct_logit_contract_v1",
    "status": "locked", "teacher": "T3 SigLIP adapted",
    "student": "S3 TinyViT-5M", "vocabulary_size": 7456,
    "vocabulary_sha256": sha256(VOCAB_PATH),
    "special_token_ids": vocab["special_token_ids"],
    "temperature": 4.0, "masking": "non-padding target positions",
    "teacher_logit_dimension": 7456, "student_logit_dimension": 7456,
    "direct_logit_kd_allowed": True,
    "test_set_status": "sealed", "test_set_used": False,
}
write_json(WORK_ROOT / "DIRECT_LOGIT_CONTRACT.json", DIRECT_LOGIT_CONTRACT)
RESULT = {
    "schema_version": "f30k_t3_adapt42_full_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete", "experiment_id": EXPERIMENT_ID,
    "teacher": "SigLIP ViT-B/16 + Flickr30k decoder",
    "seed": 42, "epochs_completed": len(history),
    "best_epoch": int(best_payload["epoch"]),
    "best_validation_metrics": best_payload["metrics"],
    "selection_metric": "Validation CIDEr-D",
    "vocabulary_size": 7456, "logit_shape_verified": True,
    "direct_logit_kd_ready": True,
    "teacher_checkpoint_sha256": sha256(CHECKPOINT_PATH),
    "best_checkpoint_sha256": sha256(BEST),
    "last_checkpoint_sha256": sha256(LAST),
    "next_phase_authorized": "F30K_T3S3_LogitKD_42",
    "test_set_status": "sealed", "test_set_used": False,
}
write_json(WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json", RESULT)
artifact_paths = sorted(
    p for p in WORK_ROOT.rglob("*") if p.is_file() and p.name != "MANIFEST.json"
)
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3_adapt42_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": EXPERIMENT_ID, "status": "complete",
    "artifacts": [
        {"relative_path": p.relative_to(WORK_ROOT).as_posix(),
         "size_bytes": p.stat().st_size, "sha256": sha256(p)}
        for p in artifact_paths
    ],
    "test_set_status": "sealed", "test_set_used": False,
})
print("=" * 78)
print(EXPERIMENT_ID, "— FINAL")
print("=" * 78)
print("Status                  : COMPLETE")
print("Best epoch             :", RESULT["best_epoch"])
print("Best Validation        :", RESULT["best_validation_metrics"])
print("Logit dimension        : 7,456")
print("Direct Logit-KD        : READY")
print("Next                   :", RESULT["next_phase_authorized"])
print("Held-out Test          : SEALED / NOT READ")
print("Output                 :", WORK_ROOT)


epoch=01 | step=0100/454 | CE=8.802136
epoch=01 | step=0200/454 | CE=7.110874
epoch=01 | step=0300/454 | CE=6.272969
epoch=01 | step=0400/454 | CE=5.749117


PTBTokenizer tokenized 74030 tokens at 274213.00 tokens per second.
PTBTokenizer tokenized 13947 tokens at 103673.40 tokens per second.


{'testlen': 11948, 'reflen': 11385, 'guess': [11948, 10934, 9920, 8906], 'correct': [7071, 3101, 1190, 469]}
ratio: 1.0494510320596355
Epoch 01/20 | train CE=5.534397 | val CE=4.078321 | CIDEr-D=0.294976 | stale=0/4
epoch=02 | step=0100/454 | CE=3.781178
epoch=02 | step=0200/454 | CE=3.705454
epoch=02 | step=0300/454 | CE=3.634103
epoch=02 | step=0400/454 | CE=3.579931


PTBTokenizer tokenized 74030 tokens at 331862.75 tokens per second.
PTBTokenizer tokenized 12618 tokens at 108820.14 tokens per second.


{'testlen': 10578, 'reflen': 10514, 'guess': [10578, 9564, 8550, 7536], 'correct': [7524, 3772, 1711, 775]}
ratio: 1.0060871219325656
Epoch 02/20 | train CE=3.555059 | val CE=3.511080 | CIDEr-D=0.510922 | stale=0/4
epoch=03 | step=0100/454 | CE=3.265041
epoch=03 | step=0200/454 | CE=3.226181
epoch=03 | step=0300/454 | CE=3.203872
epoch=03 | step=0400/454 | CE=3.180708


PTBTokenizer tokenized 74030 tokens at 358502.54 tokens per second.
PTBTokenizer tokenized 13012 tokens at 116738.44 tokens per second.


... (31 outputs left out)

PTBTokenizer tokenized 74030 tokens at 308754.98 tokens per second.
PTBTokenizer tokenized 13229 tokens at 129168.71 tokens per second.


{'testlen': 11173, 'reflen': 10966, 'guess': [11173, 10159, 9145, 8131], 'correct': [8367, 4655, 2333, 1122]}
ratio: 1.0188765274483842
Epoch 19/20 | train CE=2.354159 | val CE=2.688319 | CIDEr-D=0.766785 | stale=1/4
epoch=20 | step=0100/454 | CE=2.361117
epoch=20 | step=0200/454 | CE=2.357058
epoch=20 | step=0300/454 | CE=2.360768
epoch=20 | step=0400/454 | CE=2.361014


PTBTokenizer tokenized 74030 tokens at 367317.89 tokens per second.
PTBTokenizer tokenized 13059 tokens at 127278.84 tokens per second.


{'testlen': 11001, 'reflen': 10805, 'guess': [11001, 9987, 8973, 7959], 'correct': [8280, 4611, 2328, 1130]}
ratio: 1.0181397501155929
Epoch 20/20 | train CE=2.358931 | val CE=2.689697 | CIDEr-D=0.767916 | stale=2/4
F30K_T3_Adapt42_Full — FINAL
Status                  : COMPLETE
Best epoch             : 18
Best Validation        : {'ce_loss': 2.692232703867799, 'BLEU_1': 0.7482968806022994, 'BLEU_4': 0.33184917289858845, 'METEOR': 0.24834733383243685, 'ROUGE_L': np.float64(0.5400944778177835), 'CIDEr_D': np.float64(0.7715721111854749), 'epoch': 18, 'train_ce': 2.355422879817856, 'global_step': 8172, 'elapsed_hours': 1.3989524874422286}
Logit dimension        : 7,456
Direct Logit-KD        : READY
Next                   : F30K_T3S3_LogitKD_42
Held-out Test          : SEALED / NOT READ
Output                 : /kaggle/working/F30K_T3_Adapt42_Full
